# espmap — electrostatic potential maps in Colab

<a href="https://colab.research.google.com/github/Azuromalachite/espmap/blob/bench/timing/notebooks/espmap_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SMILES → structure check → DFT → electrostatic potential on the ρ = 0.002 e·bohr⁻³ surface → an image with a colour scale chosen from the data.

**Reading the colours:** warm/red = negative = electron-rich; cool/blue = positive = electron-poor. This is the chemistry convention.

**Status:** demonstration of research code. Sections 1–2 run on any runtime (CPU is fine for small molecules). Section 3 is an *experimental* GPU check and needs a GPU runtime (Runtime → Change runtime type → L4 or A100).

What the numbers are worth is recorded in the repo's [`README.md`](https://github.com/Azuromalachite/espmap/blob/master/README.md): e.g. def2-SVP shifts the chlorobenzene σ-hole by ~3.5 kcal/mol relative to def2-TZVPP. Pictures from the fast settings here are fine for teaching; quote numbers only from def2-TZVP or larger.

In [ ]:
#@title Setup (≈2 min)
BRANCH = "bench/timing"  #@param {type:"string"}
!git clone -q --depth 1 -b {BRANCH} https://github.com/Azuromalachite/espmap.git /content/espmap 2>/dev/null || git -C /content/espmap pull -q
!pip install -q -e /content/espmap
%cd /content/espmap
import sys; sys.path.insert(0, "/content/espmap")
import pyscf, rdkit
print("pyscf", pyscf.__version__, "| rdkit", rdkit.__version__)

In [ ]:
#@title Self-test: physics + colour-map orientation (≈10 s)
#@markdown If this fails, stop: something fundamental is wrong with the install.
!python esp_map.py --selftest

## 1 · One molecule

The structure check runs **before** any computation. Read it: it flags undrawn E/Z double bonds, unspecified stereocentres, wrong charges, disconnected fragments and reachable tautomers. It checks that a structure is *unambiguous*, not that it is the molecule you meant.

Rough costs on a 16-thread laptop CPU, def2-SVP + density fitting: chlorobenzene ~10 s, decanoic acid ~50 s.

In [ ]:
#@title Structure check
SMILES = "CCCCCCCCCC(=O)O"  #@param {type:"string"}
!python esp_map.py --smiles "{SMILES}" --check

In [ ]:
#@title Compute and render
BASIS = "def2-svp"  #@param ["def2-svp", "def2-tzvp"]
SOLVENT = "vacuum"  #@param ["vacuum", "water", "hexane", "chloroform", "toluene", "methanol", "acetonitrile", "dmso"]
DENSITY_FIT = True  #@param {type:"boolean"}
#@markdown Colour range ±LIM kcal/mol. 0 = auto-scale to this molecule (not comparable across molecules).
LIM = 0  #@param {type:"number"}
from IPython.display import Image, display
flags = f"--basis {BASIS} --solvent {SOLVENT}"
flags += " --density-fit" if DENSITY_FIT else ""
flags += f" --lim {LIM}" if LIM else ""
!python esp_map.py --smiles "{SMILES}" {flags} --yes --out /content/esp.png
display(Image("/content/esp.png", width=500))

## 2 · A comparable series

Panels are only comparable when they share **one** colour range. Each molecule is run with the same `--lim`; check the printed `V = … kcal/mol` lines — if any value exceeds ±LIM, raise it and rerun.

The default series is neutral, so every surface changes sign and a symmetric diverging scale is correct. **Do not mix in ions** (e.g. acetate): an ion's surface never changes sign, and one shared symmetric scale cannot show both honestly (see `docs/roadmap-review.md`, F2).

In [ ]:
#@title Run the series (≈1 min on CPU)
SERIES = "ethanol:CCO, acetaldehyde:CC=O, acetic acid:CC(=O)O, ethyl acetate:CCOC(C)=O"  #@param {type:"string"}
SHARED_LIM = 60  #@param {type:"number"}
import subprocess, matplotlib.pyplot as plt, matplotlib.image as mpimg
items = [s.split(":", 1) for s in SERIES.split(",")]
fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4.4))
for i, (ax, (name, smi)) in enumerate(zip(list(axes) if len(items) > 1 else [axes], items)):
    name, smi, out = name.strip(), smi.strip(), f"/content/series_{i}.png"
    p = subprocess.run(["python", "esp_map.py", "--smiles", smi, "--basis", "def2-svp",
                        "--density-fit", "--lim", str(SHARED_LIM), "--yes", "--out", out],
                       capture_output=True, text=True)
    vline = next((l for l in p.stdout.splitlines() if l.startswith("V = ")), p.stderr[-300:])
    print(f"{name:15s} {vline}")
    if p.returncode == 0:
        ax.imshow(mpimg.imread(out))
    ax.set_title(name); ax.axis("off")
fig.suptitle(f"B3LYP/def2-SVP (density-fitted), shared scale ±{SHARED_LIM} kcal/mol, warm = negative")
plt.tight_layout(); plt.show()

## 3 · Experimental: does a GPU reproduce the CPU numbers?

Needs a GPU runtime. This uses [GPU4PySCF](https://github.com/pyscf/gpu4pyscf) for the SCF only; surface and ESP stay on the CPU. It is **not** part of the espmap pipeline yet: it re-runs one row of the CPU benchmark (`bench/timing.py`) and compares.

Pass condition: total energy within 1×10⁻⁵ Ha and V_min / V_max within 0.05 kcal/mol of the CPU reference. The comparison is only like-for-like if the geometry hash matches; a different RDKit build can embed a slightly different conformer.

Note that DFT needs double precision, which the L4 and T4 run slowly. An L4 may be no faster than a recent laptop CPU; an A100 should be much faster. This cell measures it rather than assuming.

In [ ]:
#@title Install GPU4PySCF (GPU runtime only)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: switch runtime type first"
!pip install -q gpu4pyscf-cuda12x cutensor-cu12
import pyscf, gpu4pyscf
print("pyscf", pyscf.__version__, "| gpu4pyscf", gpu4pyscf.__version__)

In [ ]:
#@title Decanoic acid, B3LYP/def2-TZVP, density-fitted: GPU vs CPU reference
import hashlib, time
import numpy as np
from pyscf import dft, gto
import esp_map as em

# from bench/results.jsonl, row decanoic_acid|def2-tzvp|df|16t
# (Ryzen 7 260, 16 threads, PySCF 2.14.0, RDKit 2026.09.1)
REF = dict(geom_sha="26dcaf3519", nao=492, e_tot=-543.8103149269617,
           V_min=-37.277894106462526, V_max=60.079848397342694, t_scf=97.4)

spec, _ = em.from_smiles("CCCCCCCCCC(=O)O")
sha = hashlib.sha1(spec.encode()).hexdigest()[:10]
mol = gto.M(atom=spec, basis="def2-tzvp", unit="Angstrom", verbose=0)
# mirrors esp_map.run_scf (RKS, b3lyp, grids.level 3, density fitting);
# if this validates, the GPU switch belongs inside run_scf, not here
mf = dft.RKS(mol).density_fit()
mf.xc = "b3lyp"
mf.grids.level = 3
mf = mf.to_gpu()
t = time.perf_counter(); mf.kernel(); t_scf = time.perf_counter() - t
assert mf.converged, "GPU SCF did not converge"
dm = em.total_dm(mf)
dm = dm.get() if hasattr(dm, "get") else np.asarray(dm)   # cupy -> numpy
verts, faces, diag = em.extract_surface(mol, dm, 0.26)
V = em.esp_at_points(mol, dm, verts) * em.HA2KCAL

got = dict(geom_sha=sha, nao=mol.nao, e_tot=float(mf.e_tot),
           V_min=float(V.min()), V_max=float(V.max()), t_scf=t_scf)
tol = dict(e_tot=1e-5, V_min=0.05, V_max=0.05)
print(f"{'':10s} {'CPU ref':>18s} {'GPU':>18s} {'diff':>12s}")
for k in REF:
    r, g = REF[k], got[k]
    if isinstance(r, str) or k == "nao":
        print(f"{k:10s} {r!s:>18s} {g!s:>18s} {'same' if r == g else 'DIFFERENT':>12s}")
    else:
        d = g - r
        flag = "" if k not in tol else ("  ok" if abs(d) <= tol[k] else "  FAIL")
        print(f"{k:10s} {r:18.6f} {g:18.6f} {d:+12.6f}{flag}")
print(f"\nSCF speed-up over the 16-thread laptop CPU: {REF['t_scf'] / t_scf:.1f}x")
if sha != REF["geom_sha"]:
    print("WARNING: different geometry from the reference; the comparison is not like-for-like.")

## Limits worth knowing

- **One conformer.** SMILES input uses the lowest-energy MMFF conformer. ESP extrema are fairly robust to conformation; the *shape* of the surface is not.
- **Validated elements only.** Excited states, periodic systems and transition metals are out of scope.
- **Fluorine shows no σ-hole** because it does not form one; that is deliberate.
- **Protonation is your choice.** A carboxylic acid at pH 7 is mostly carboxylate, and the ESP changes completely. Draw the state you mean.

Source, measured error budget and known defects: <https://github.com/Azuromalachite/espmap>